# PROBLEMA 2 - INTER 2 Probabilidad y Estadística - iMAT-ICAI - 7 de marzo 2024

# Vale el 20% de la nota del examen
---

*NO CAMBIES EL NOMBRE AL NOTEBOOK, SIMPLEMENTE ESCRIBE AQUÍ TUS DATOS*
### Nombre: RELLENAR
### Apellidos: RELLENAR
---

### Se quiere modelar la distribución de probabilidad de una v.a. continua X. Dicha variable X recoge la probabilidad de que ocurra un determinado evento, por lo que toma valores entre 0 y 1. 

### Se dispone de una muestra de datos de X, a partir de la que se quieren estimar por máxima verosimilitud los parámetros de una distribución beta. La distribución beta se caracteríza por tener bastante flexibilidad para modelar diferentes formas de funciones de densidad de probabilidad, en el rango de valores de la variable entre 0 y 1, ideal para lo que se quiere conseguir.

Responde las preguntas utilizando instrucciones print() para mostrar claramente tu respuesta en la ejecución de la celda, explicando cómo lo resuelves. Utiliza comentarios en tu código para facilitar su lectura.

### Importamos todos los paquetes aquí por comodidad

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import seaborn as sns
from scipy import optimize as opt  # para estimación por máxima verosimilitud

### Cargamos la muestra de datos está en el fichero "muestra_PROB2_INTER2024.csv", conteniendo la variable "x".

In [ ]:
# cargamos los datos de la muestra disponible
x = np.loadtxt('muestra_PROB2_INTER2024.csv', delimiter=',', skiprows=1) #   <-----   x (datos de la muestra)

# tamaño muestral
n = len(x)                                                         #   <-----   n (tamaño de la muestra)
print(f'Muestra cargada con {n} observaciones')

fig = plt.figure(1, figsize=(12, 4)) # permite indicar el nº de la figura y las dimensiones (ancho y alto)
plt.hist(x, bins=20, density = True)
sns.rugplot(x, height = 0.025, lw = 0.4, color = 'red')
plt.title(f'Histograma muestra')
plt.xlabel('x')
plt.ylabel('Frecuencia relativa')
plt.show()

### La distribución beta tiene dos parámetros (alfa y beta, denominados "a" y "b" cuando se crea una v.a.) que permiten cambiar la forma de la función de densidad de probabilidad con bastante libertad. A continuación se muestran unos ejemplos:

In [ ]:
# puntos en los que se evalua las funciones para representarlas
xs = np.linspace(-0.1,1.1, 1000)

# pinta sincronizadamente ambas funciones
fig = plt.figure(1, figsize=(15, 4))

VA = stats.beta(a=0.9,b=0.9)
plt.plot( xs, VA.pdf(xs), label = 'pdf beta(a=0.9,b=0.9)')
VA = stats.beta(a=1,b=2)
plt.plot( xs, VA.pdf(xs), label = 'pdf beta(a=1,b=2)')
VA = stats.beta(a=2,b=2)
plt.plot( xs, VA.pdf(xs), label = 'pdf beta(a=2,b=2)')
VA = stats.beta(a=3,b=3)
plt.plot( xs, VA.pdf(xs), label = 'pdf beta(a=3,b=3)')
VA = stats.beta(a=3,b=1)
plt.plot( xs, VA.pdf(xs), label = 'pdf beta(a=3,b=1)')
VA = stats.beta(a=4,b=3)
plt.plot( xs, VA.pdf(xs), label = 'pdf beta(a=4,b=3)')

plt.xlabel('x')
plt.ylabel('p')
plt.legend()
plt.grid()
plt.show()

### a) Ajusta por máxima verosimilitud los parámetros alfa y beta de la distribución beta utilizando la muestra data. Justifica la respuesta. Asegúrate de que la optimización acaba encontrando una solución. ¿Son iguales los parámetros alfa y beta estimados? (3 puntos)

In [ ]:
# Respuesta

# Definimos la función de verosimilitud logarítmica (* -1 para maximizar luego)

def verosimilitud_log(parameters):
    # sacamos los parámetros
    param1, param2 = parameters
    
    # La función de verosimilitud logarítmica depende del problema y requiere
    # conocer la distribución de la v.a. bajo estudio, que dependerá de los params
    # a estimar por MV
    
    # Ejemplo estimando la media y desv. típica de una normal:
    logLikelihood = np.sum(stats.beta.logpdf(x, a=param1, b=param2))  # <--- logpdf es el log de la pdf, ya está preparado ...
    
    # Como la función minimiza y queremos los params que maximizan la verosimilitud
    # simplemente la multimplicamos por -1: max(loglike) = min(-loglike)
    neg_logLikelihood = -1 * logLikelihood
    return neg_logLikelihood

In [ ]:
# Maximizamos la función de verosimilitud logarítmica (minimizando -1 * funcion)
mle_model = opt.minimize(verosimilitud_log, np.array([1,1]), method='L-BFGS-B')

ver_log_beta1 = -1 * mle_model.fun

# cogemos los coefs estimados
param1 = mle_model.x[0]
param2 = mle_model.x[1]

print(mle_model)
print('Parámetros estimados por MV:\n'
      f'  - parámetro 1: {param1}\n'
      f'  - parámetro 2: {param2}')

print(f'Se puede observar que alfa y beta se parecen, pero no son exactamente iguales.')

### b) Crea una v.a. distribuida según la beta con los parámetros estimados y representa en la misma gráfica la función de densidad dicha distribución, junto con el histograma de la muestra. Recuerda representar el histograma con la opción "density = True" para facilitar la comparación visual. ¿Te parece que los datos de la muestra podrían seguir esa distribución? Justifica la respuesta. (1 punto)

In [ ]:
# Respuesta

# crea la variable
alfa1 = param1
beta1 = param2
VA1 = stats.beta(a=alfa1, b=beta1)

# representa
fig = plt.figure(1, figsize=(12, 4))
plt.plot( xs, VA1.pdf(xs), label = f'pdf beta(a={alfa1},b={beta1})')
plt.hist(x, bins=20, density = True, label = 'Histograma muestra')
sns.rugplot(x, height = 0.025, lw = 0.4, color = 'red', label = 'Muestra')
plt.title(f'Histograma de la muestra y distribución estimada por MV')
plt.xlabel('x')
plt.ylabel('Frecuencia relativa')
plt.legend()
plt.show()

### c) Se sospecha que la distribución real es simétrica. Bajo esta hipotesis, repite el ajuste por máxima verosimilitud forzando a que los parámetros alfa y beta de la distribución beta sean iguales. ¿Han cambiado por igual ambos parámetros? Justifica la respuesta. Asegúrate de que la optimización acaba encontrando una solución. (3 puntos)

In [ ]:
# Respuesta

# Definimos la nueva función de verosimilitud logarítmica

print('Ahora la función de verosimilitud logarítimica recibe un único parámetro, que será el mismo para alfa y beta.')

def verosimilitud_beta_simetrica_log(parameters):
    # sacamos los parámetros
    param1 = parameters
    
    # La función de verosimilitud logarítmica depende del problema y requiere
    # conocer la distribución de la v.a. bajo estudio, que dependerá de los params
    # a estimar por MV
    
    # Ejemplo estimando la media y desv. típica de una normal:
    logLikelihood = np.sum(stats.beta.logpdf(x, a=param1, b=param1))  # <--- logpdf es el log de la pdf, ya está preparado ...
    
    # Como la función minimiza y queremos los params que maximizan la verosimilitud
    # simplemente la multimplicamos por -1: max(loglike) = min(-loglike)
    neg_logLikelihood = -1 * logLikelihood
    return neg_logLikelihood

In [ ]:
# Maximizamos la función de verosimilitud logarítmica (minimizando -1 * funcion)
mle_model = opt.minimize(verosimilitud_beta_simetrica_log, np.array([1]), method='L-BFGS-B')

# cogemos los coefs estimados
param1 = mle_model.x[0]

ver_log_beta2 = -1 * mle_model.fun

print(mle_model)
print('Parámetro estimado por MV:\n'
      f'  - parámetro 1: {param1}')

print(f'Ahora alfa = beta = {param1}')
print('Se podría haber pensado que al imponer que alfa y beta fuesen iguales, el valor obtenido sería '
      'uno de compromiso entre los dos valores estimados previamente, pero no ha sido así.')

dif_alfas = alfa1 - param1
dif_betas = beta1 - param1

print('Si medimos la diferencia:')
print(f'alfa1 - alfa2 = {alfa1} - {param1} = {dif_alfas}')
print(f'beta1 - beta2 = {beta1} - {param2} = {dif_betas}')
print('Se confirma que el parámetro beta cambia más que el alfa de una estimación a la otra.')

### d) Crea una nueva v.a. distribuida según la beta estimada imponiendo que los dos parámetros son iguales y representa en la misma gráfica las funciones de densidad de las distribuciones beta estimadas en los dos apartados anteriores. Incluye también el histograma de la muestra para comparar.  ¿Hay mucha diferencia? ¿Tiene sentido plantear que la distribución teórica sea simétrica? Justifica la respuesta. (1.5 puntos)

In [ ]:
# Respuesta

# crea la variable
alfa2 = param1
beta2 = param1
VA2 = stats.beta(a=alfa2, b=beta2)

# representa
fig = plt.figure(1, figsize=(12, 4))
plt.plot( xs, VA1.pdf(xs), label = f'pdf beta(a={alfa1},b={beta1})')
plt.plot( xs, VA2.pdf(xs), label = f'pdf beta(a={alfa2},b={beta2})')
plt.hist(x, bins=20, density = True)
sns.rugplot(x, height = 0.025, lw = 0.4, color = 'red')
plt.title(f'Histograma de la muestra y distribución estimada por MV')
plt.xlabel('x')
plt.ylabel('Frecuencia relativa')
plt.legend()
plt.show()

print('Viendo los resultados, sí tiene sentido pensar que la distribución teórica pueda ser simétrica ya que',
      'La distribución estimada inicialmente con libertad para encontrar cualquier pareja de parámetros se parece',
      'mucho a la obtenida forzando a que sean iguales.')

### e) De las dos distribuciones beta ajustadas en los apartados anteriores, ¿cuál hace que la muestra de datos sea más verosímil? Justifica la respuesta. (1.5 puntos)

In [ ]:
# Respuesta

print('Recuperamos las verosimilitudes logáritmicas calculadas según ambas distribuciones beta:')
print(f'beta(a={alfa1},b={beta1}) -> verosimilitud logarítmica = {ver_log_beta1}')
print(f'beta(a={alfa2},b={beta2}) -> verosimilitud logarítmica = {ver_log_beta2}')

print('Dado que la verosimilitud de la primera distribución beta es un poco mayor que la segunda, se puede '
      'decir que, como era de esperar, la primera beta (sin imponer parámetros iguales) es la que hace más verosímil la muestra. '
      'No es raro este resultado ya que al tener más grados de libertad puede adaptarse mejor a esa muestra concreta, aunque pueda '
      'ser menos correcta para caracterizar a la población si es realmente simétrica.')